# Vision Transformers — transfer learning

**DeepVision Masterclass — Modern Edition**

This notebook is designed as an executable teaching lab. Read the dataset contract and validation notes before training.

## 1. From CNNs to ViTs
Vision Transformers split an image into patches, embed them as tokens, and use self-attention. Pretrained weights are strongly recommended for small datasets.

In [ ]:
import torch, torchvision
print(torch.__version__,torchvision.__version__)
from torchvision.models import vit_b_16, ViT_B_16_Weights


## 2. Load pretrained ViT-B/16
The weights object supplies preprocessing. ViT-B/16 expects 224×224 inputs and 16×16 patches.

In [ ]:
weights=ViT_B_16_Weights.DEFAULT
model=vit_b_16(weights=weights)
preprocess=weights.transforms()
for p in model.parameters(): p.requires_grad=False
in_features=model.heads.head.in_features
model.heads.head=torch.nn.Linear(in_features,10)
for p in model.heads.head.parameters(): p.requires_grad=True
print('Trainable parameters:',sum(p.numel() for p in model.parameters() if p.requires_grad))

## 3. Fine-tuning strategy
Train the new head first, then unfreeze selected encoder blocks with a smaller learning rate. ViTs can overfit quickly on small datasets.

In [ ]:
# Unfreeze the final encoder block after head training:
for p in model.encoder.layers[-1].parameters(): p.requires_grad=True
optimizer=torch.optim.AdamW(filter(lambda p:p.requires_grad,model.parameters()),lr=1e-5,weight_decay=.05)
# Use a validation set, warmup and cosine decay for a more rigorous experiment.

## 4. Attention and patch intuition
For 224×224 with patch size 16, there are 14×14=196 image patches plus a class token. Attention cost grows quadratically with token count.

In [ ]:
image_size,patch_size=224,16
num_patches=(image_size//patch_size)**2
print('Patch tokens:',num_patches,'with CLS token:',num_patches+1)

## 5. Evaluate fairly
Compare a pretrained CNN (ResNet50) and ViT under the same data split, image resolution, augmentation, compute budget and checkpoint-selection rule.

## 6. Exercises
Compare ViT-B/16 with Swin Transformer and ConvNeXt; inspect attention maps carefully; evaluate robustness to blur, corruption and distribution shift; benchmark CPU/GPU latency.